<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Modèle de données SQL

Objectif : à partir de la table `inspections` nettoyée, construire un **panel pont × année** avec les événements de dégradation, plus les tables de dimensions.

- **Partie 0** (fournie) : setup + chargement/nettoyage → table `inspections`
- **Étapes 1 à 7** : c'est toi qui écris le SQL

## 0.1 — Setup

In [1]:
!pip install -q jupysql duckdb-engine

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

Mounted at /content/drive


## 0.2 — Chargement + nettoyage → `inspections`

Une ligne par pont × année (2016–2025 ; le fichier 2015 téléchargé n'est pas exploitable).

| Colonne | Contenu |
|---|---|
| `bridge_id` | `state_code` + `-` + `structure_number` |
| `year` | année du fichier NBI |
| `state_code`, `county_code`, `lat`, `lon` | localisation (lat/lon en degrés décimaux, lon négative) |
| `owner_code`, `material_code`, `structure_type_code` | codes bruts (items 22, 43A, 43B) |
| `year_built`, `year_reconstructed` | entiers, NULL si 0/absent |
| `adt`, `pct_truck`, `length_m`, `deck_width_m` | trafic et géométrie |
| `deck`, `super`, `sub`, `culvert` | notes 0–9, `'N'` → NULL |
| `is_culvert` | `culvert IS NOT NULL` |
| `min_rating` | `culvert` si ponceau, sinon `LEAST(deck, super, sub)` |
| `age` | `year - year_built` |

Choix de nettoyage :
- on garde seulement `RECORD_TYPE_005A = '1'` (les lignes de type 2 décrivent la route *sous* le pont et dupliqueraient la structure) ;
- les doublons éventuels ne sont **pas** supprimés ici : tu les chercheras à l'étape 6.

⏱️ Lecture de ~2,5 Go de CSV : compte quelques minutes.

In [2]:
%%sql
-- Chargement + nettoyage en une seule passe : on lit directement les CSV bruts
-- et on ne garde que les colonnes utiles, déjà typées.
CREATE OR REPLACE TABLE inspections AS
WITH raw AS (
    SELECT
        CAST(regexp_extract(filename, 'nbi_(20\d{2})', 1) AS INTEGER) AS year,
        *
    FROM read_csv(
        [   -- 2016 à 2025 : nbi_2015.txt n'est pas un vrai fichier NBI (125 lignes)
            '/content/drive/MyDrive/nbi/raw/nbi_201[6-9].txt',
            '/content/drive/MyDrive/nbi/raw/nbi_202[0-9].txt'
        ],
        encoding = 'latin-1',
        delim = ',',
        quote = '''',            -- le qualificateur de texte NBI est l'apostrophe
        header = true,
        all_varchar = true,      -- tout en texte, on type nous-mêmes ci-dessous
        union_by_name = true,
        filename = true,
        strict_mode = false      -- tolère les apostrophes dans les textes (ex. 'O'BRIEN CREEK')
    )
    WHERE RECORD_TYPE_005A = '1'   -- 1 = la structure elle-même (2 = route qui passe dessous, doublon)
),
typed AS (
    SELECT
        lpad(trim(STATE_CODE_001), 2, '0') || '-' || trim(STRUCTURE_NUMBER_008) AS bridge_id,
        year,
        lpad(trim(STATE_CODE_001), 2, '0')  AS state_code,
        lpad(trim(COUNTY_CODE_003), 3, '0') AS county_code,

        -- LAT_016 = DDMMSSss, LONG_017 = DDDMMSSss (ouest, donc négatif)
        lpad(trim(LAT_016), 8, '0')  AS lat_txt,
        lpad(trim(LONG_017), 9, '0') AS lon_txt,

        lpad(trim(OWNER_022), 2, '0')            AS owner_code,
        trim(STRUCTURE_KIND_043A)                AS material_code,
        lpad(trim(STRUCTURE_TYPE_043B), 2, '0')  AS structure_type_code,

        NULLIF(TRY_CAST(YEAR_BUILT_027         AS INTEGER), 0) AS year_built,
        NULLIF(TRY_CAST(YEAR_RECONSTRUCTED_106 AS INTEGER), 0) AS year_reconstructed,

        TRY_CAST(ADT_029               AS INTEGER) AS adt,
        TRY_CAST(PERCENT_ADT_TRUCK_109 AS INTEGER) AS pct_truck,
        TRY_CAST(STRUCTURE_LEN_MT_049  AS DOUBLE)  AS length_m,
        TRY_CAST(DECK_WIDTH_MT_052     AS DOUBLE)  AS deck_width_m,

        -- notes 0-9 ; 'N' (non applicable) ou vide -> NULL
        TRY_CAST(DECK_COND_058           AS TINYINT) AS deck,
        TRY_CAST(SUPERSTRUCTURE_COND_059 AS TINYINT) AS super,
        TRY_CAST(SUBSTRUCTURE_COND_060   AS TINYINT) AS sub,
        TRY_CAST(CULVERT_COND_062        AS TINYINT) AS culvert
    FROM raw
)
SELECT
    bridge_id, year, state_code, county_code,
    CASE WHEN TRY_CAST(lat_txt AS BIGINT) > 0 THEN
        CAST(substr(lat_txt, 1, 2) AS INTEGER)
      + CAST(substr(lat_txt, 3, 2) AS INTEGER) / 60.0
      + CAST(substr(lat_txt, 5, 4) AS INTEGER) / 100.0 / 3600.0
    END AS lat,
    CASE WHEN TRY_CAST(lon_txt AS BIGINT) > 0 THEN
      -( CAST(substr(lon_txt, 1, 3) AS INTEGER)
       + CAST(substr(lon_txt, 4, 2) AS INTEGER) / 60.0
       + CAST(substr(lon_txt, 6, 4) AS INTEGER) / 100.0 / 3600.0 )
    END AS lon,
    owner_code, material_code, structure_type_code,
    year_built, year_reconstructed,
    adt, pct_truck, length_m, deck_width_m,
    deck, super, sub, culvert,
    culvert IS NOT NULL AS is_culvert,
    CASE WHEN culvert IS NOT NULL THEN culvert
         ELSE LEAST(deck, super, sub)          -- LEAST ignore les NULL dans DuckDB
    END AS min_rating,
    year - year_built AS age
FROM typed;

Running query in 'DuckDBPyConnection'

InvalidInputException: Invalid Input Error: CSV Error on Line: 1
Original Line: 01,012747         ,1,5,0,00000,0,06,081,57048,'NS RAILROAD',,'AVE "B"','OPELIKA',99.99,0.1,0,000MU00020,0,32080000,085270000,2,3,04,04,16,1982,4,0,6406,2018,5,15.8,0,0,0,1,1,1,1,5,N,0,0,A,1,2,4,02,0,00,3,0,18.9,30.8,61.4,1.5,1.5,15.8,19.8,99.99,R,7.01,R,4,4,7,7,7,N,N,1,43,1,25.8,6,4,5,5,N,6,,,71.8,918,24,N  ,N  ,N  ,,,,3152,315,3467,2019,,0,_              ,0,N,2,,0,0,0,1,1,0,0,5,0,,Y,N,9120,2038,0,N,01,G,7,1215.72
01,012748         ,1,4,1,00003,0,06,101,00000,'BLACKS SLOUGH',,'WOODLEY ROAD','21.6 MI N OF PIKE CO LINE',99.99,0.01,0,000CO00030,0,32150579,086122093,0,3,02,02,08,1993,2,0,900,1988,5,11,0,0,0,0,0,0,1,5,N,0,0,A,1,9,1,22,0,00,1,0,8.6,10.4,10.4,0,0,8.6,9.4,99.99,N,0,N,0,0,7,7,7,7,N,1,74.1,1,44.5,7,5,N,5,7,8,,,17.1,1217,24,N  ,N  ,N  ,,,,352,35,387,2019,,0,_              ,0,N,2,,0,0,0,2,1,0,0,1,0,,Y,8,1200,2037,0,N,01,G,7,97.76
01,012749         ,1,4,1,00003,0,06,101,00000,'DRAINAGE RELIEF',,'WOODL

In [3]:
%%sql
-- Vérification rapide du chargement (pas l'étape 1 : juste s'assurer que la table existe)
SELECT COUNT(*) AS n_rows, MIN(year) AS first_year, MAX(year) AS last_year
FROM inspections;

Running query in 'DuckDBPyConnection'

BinderException: Binder Error: Referenced column "year" not found in FROM clause!
Candidate bindings: "deck"

LINE 1: SELECT COUNT(*) AS n_rows, MIN(year) AS first_year, MAX(year) AS last_year
                                       ^

In [ ]:
%%sql
SELECT * FROM inspections LIMIT 5;

---
## Étape 1 — Prise en main de la table (15 min)

**Objectif :** répondre à 4 questions, une requête chacune.
1. Combien de lignes (= ponts inspectés) par année ?
2. Combien de ponts **distincts** au total sur toute la période ?
3. Distribution de `min_rating` par année (combien de ponts à 0, 1, …, 9 chaque année).
4. Part de ponceaux (`is_culvert`) — globale ou par année.

**Concepts :** `GROUP BY`, `COUNT(DISTINCT …)`, agrégats conditionnels `COUNT(*) FILTER (WHERE …)`.

**Indices (sans solution) :**
- Q1 et Q2 ne donnent pas le même ordre de grandeur que la somme des lignes : pourquoi ?
- Q3 : deux façons possibles — format « long » (`GROUP BY` sur deux colonnes) ou format « large » (une colonne par note avec `FILTER`). Essaie au moins le format large pour pratiquer `FILTER`.
- Q4 : un booléen se compte avec `FILTER (WHERE is_culvert)`, et une part = compte filtré / compte total. Attention à la division entière.
- Pense aux `NULL` dans `min_rating` : où vont-ils dans tes résultats ?

In [ ]:
%%sql
-- Q1 — nombre de ponts par année


In [ ]:
%%sql
-- Q2 — nombre de ponts distincts


In [ ]:
%%sql
-- Q3 — distribution de min_rating par année


In [ ]:
%%sql
-- Q4 — part de ponceaux
